# Script 4 — Avaliação Aprofundada, Risco Corporativo e Análise de Estresse

**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Este script consome os artefatos do Script 3 e executa:

1. **Relatório comparativo** de métricas (RMSE, MAE, SMAPE, R², Theil-U, DA) por algoritmo, target e horizonte  
2. **Z-Score de Altman Z''** adaptado para mercados emergentes — por empresa e período (histórico + prospectivo sobre KPIs preditos)  
3. **Score de risco composto** (0–100) baseado em 8 limiares financeiros consolidados  
4. **Análise probabilística de estresse** — Monte Carlo sobre KPIs preditos  
5. **Feature Importance** agregada multi-target com ranking por família  
6. **Análise de resíduos** por empresa e horizonte  
7. **Persistência** de todos os artefatos para o Script 5 (cenários + LLM)

---
`Z'' = 6.56·X1 + 3.26·X2 + 6.72·X3 + 1.05·X4`  
Zonas: Z'' > 2.60 → Segura | 1.10–2.60 → Cinza | < 1.10 → Insolvência

## Etapa 0 — Imports, Configuração e Logging

In [ ]:
import json, logging, pickle, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 60)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
(PASTA_SAIDA / 'logs').mkdir(exist_ok=True)
(PASTA_SAIDA / 'figuras').mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_avaliacao_v4')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_avaliacao.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Constantes de transformação (espelhadas do Script 3) ──────────────────────
_TARGET_BASES = [
    'DRE_3.01', 'DRE_3.11', 'EBITDA',
    'BPA_1', 'BPA_1.01', 'BPP_2.01', 'BPP_2.03', 'BPP_2', 'DFC_MI_6.01',
]
_HORIZONTES    = ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']
_LOG_BASES     = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARCSINH_BASES = {'DFC_MI_6.01','DRE_3.11'}

LOG_TARGETS     = {f'TARGET_{b}{h}' for b in _LOG_BASES     for h in _HORIZONTES}
ARCSINH_TARGETS = {f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES}

NOME_TARGET = {
    'DRE_3.01':'Receita Líquida', 'DRE_3.11':'Lucro Líquido', 'EBITDA':'EBITDA',
    'BPA_1':'Ativo Total', 'BPA_1.01':'Ativo Circulante',
    'BPP_2.01':'Passivo Circulante', 'BPP_2.03':'Passivo Não Circulante',
    'BPP_2':'Patrimônio Líquido', 'DFC_MI_6.01':'FCO (Caixa Operacional)',
}

# Targets estratégicos do TCC
TARGETS_FOCO_TCC = [
    f'TARGET_{b}{h}'
    for b in ['DRE_3.01', 'DRE_3.11', 'EBITDA']
    for h in _HORIZONTES
]

logger.info('Script 4 — Avaliação Aprofundada iniciado em %s', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('✅ Configuração carregada')
print(f'Pasta de saída: {PASTA_SAIDA.resolve()}')

## Etapa 1 — Carga dos Artefatos do Script 3

In [ ]:
def carregar_pkl(nome, obrigatorio=True):
    path = PASTA_SAIDA / nome
    if not path.exists():
        msg = f'{nome} não encontrado — execute o Script 3 antes.'
        if obrigatorio: raise FileNotFoundError(msg)
        logger.warning(msg); return None
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    logger.info('Carregado: %s', nome)
    return obj

# ── Artefatos obrigatórios ────────────────────────────────────────────────────
treino  = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste   = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')

FEATURES                     = carregar_pkl('features.pkl')
KPIS                         = carregar_pkl('kpis.pkl', obrigatorio=False) or []
melhores                     = carregar_pkl('melhores_modelos.pkl')
metricas_teste_pkl           = carregar_pkl('metricas_teste.pkl')
baselines                    = carregar_pkl('baselines.pkl')
feature_importances          = carregar_pkl('feature_importances.pkl')
selected_features_por_target = carregar_pkl('selected_features_por_target.pkl')
TARGETS_POR_HORIZONTE        = carregar_pkl('targets_por_horizonte.pkl', obrigatorio=False) or {}

_p_cv  = PASTA_SAIDA / 'resultados_cv.csv'
_p_te  = PASTA_SAIDA / 'resultados_teste.csv'
df_cv  = pd.read_csv(_p_cv)  if _p_cv.exists()  else pd.DataFrame()
df_te  = pd.read_csv(_p_te)  if _p_te.exists()  else pd.DataFrame()

_p_pred = PASTA_SAIDA / 'predicoes_teste_detalhadas.parquet'
df_pred = pd.read_parquet(_p_pred) if _p_pred.exists() else pd.DataFrame()

TARGETS = [
    f'TARGET_{b}{h}'
    for b in _TARGET_BASES
    for h in _HORIZONTES
    if f'TARGET_{b}{h}' in melhores
]

_p_ds   = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
dataset = pd.read_parquet(_p_ds) if _p_ds.exists() else pd.concat([treino, teste], ignore_index=True)

for _df in [treino, teste, dataset]:
    for _col in ('DT_REFER','DT_TARGET','DT_TARGET_DFP'):
        if _col in _df.columns:
            _df[_col] = pd.to_datetime(_df[_col], utc=True, errors='coerce').dt.tz_localize(None)

print(f'Treino  : {treino.shape[0]:,} obs | {treino.shape[1]} colunas')
print(f'Teste   : {teste.shape[0]:,} obs  | {teste.shape[1]} colunas')
print(f'Dataset : {dataset.shape[0]:,} obs | {dataset.shape[1]} colunas')
print(f'TARGETS ativos   : {len(TARGETS)}')
print(f'Melhores modelos : {", ".join(sorted(set(melhores.values())))}')
if 'SETOR' in dataset.columns:
    print(f'Setores : {sorted(dataset["SETOR"].dropna().unique().tolist())}')
logger.info('Artefatos carregados | TARGETS=%d | Treino=%d | Teste=%d',
            len(TARGETS), len(treino), len(teste))

## Etapa 2 — Funções de Métricas e Transformações

In [ ]:
def get_target_transform(target):
    if target in LOG_TARGETS:     return 'log1p'
    if target in ARCSINH_TARGETS: return 'arcsinh'
    return 'none'

def target_inverse_transform(y_pred, transformacao='none'):
    y = np.asarray(y_pred, float)
    if transformacao == 'log1p':   return np.expm1(y)
    if transformacao == 'arcsinh': return np.sinh(y)
    return y

def rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt,float)-np.asarray(yp,float))**2)))

def mae(yt, yp):
    return float(np.mean(np.abs(np.asarray(yt,float)-np.asarray(yp,float))))

def smape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    denom  = (np.abs(yt)+np.abs(yp))/2.0
    mask   = denom > 1e-9
    return float(np.mean(np.abs(yt[mask]-yp[mask])/denom[mask])) if mask.sum()>0 else np.nan

def mape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    mask   = np.abs(yt) > 1e-9
    return float(np.mean(np.abs((yt[mask]-yp[mask])/yt[mask]))) if mask.sum()>0 else np.nan

def r2_seguro(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2 or np.isclose(np.var(yt),0): return np.nan
    ss_res = np.sum((yt-yp)**2); ss_tot = np.sum((yt-yt.mean())**2)
    return float(1-ss_res/ss_tot) if ss_tot>0 else np.nan

def theil_u(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2: return np.nan
    em = np.sqrt(np.mean((yt[1:]-yp[1:])**2))
    en = np.sqrt(np.mean((yt[1:]-yt[:-1])**2))
    return float(em/en) if en>0 else np.nan

def da_score(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    if len(yt)<2: return np.nan
    return float(np.mean(np.sign(yt[1:]-yt[:-1])==np.sign(yp[1:]-yp[:-1])))

def calcular_metricas(yt, yp):
    return {'RMSE':rmse(yt,yp),'MAE':mae(yt,yp),'SMAPE':smape(yt,yp),
            'MAPE':mape(yt,yp),'R2':r2_seguro(yt,yp),
            'TheilU':theil_u(yt,yp),'DA':da_score(yt,yp),'N':len(yt)}

print('✅ Funções de métricas definidas')

## Etapa 3 — Relatório de Métricas por Algoritmo, Target e Horizonte

In [ ]:
if not df_te.empty:
    print('=== Desempenho no Hold-out por Algoritmo e Horizonte ===')

    df_foco = df_te[df_te['Target'].isin(TARGETS_FOCO_TCC)].copy()
    if df_foco.empty: df_foco = df_te.copy()

    metricas_cols = [c for c in [
        'SMAPE_teste_macro_empresa','RMSE_teste_macro_empresa',
        'R2_teste_macro_empresa','TheilU_teste_macro_empresa','DA_teste_macro_empresa'
    ] if c in df_foco.columns]

    if metricas_cols and 'Horizonte' in df_foco.columns and 'Algoritmo' in df_foco.columns:
        grp = df_foco.groupby(['Horizonte','Algoritmo'])[metricas_cols].mean()
        print('\nMédias por Horizonte × Algoritmo (DRE_3.01, DRE_3.11, EBITDA):')
        print(grp.round(4).to_string())

    print('\n=== Contagem de Melhores Modelos por Target (critério SMAPE_CV) ===')
    cnt = Counter(melhores.values())
    for alg, n in sorted(cnt.items(), key=lambda x: -x[1]):
        pct = n/len(melhores)*100
        print(f'  {alg:<22} {n:>3}× ({pct:5.1f}%) {"█"*int(pct/3)}')

    if metricas_cols and 'Horizonte' in df_te.columns:
        print('\n=== SMAPE médio por horizonte × algoritmo (todos os targets) ===')
        _c = metricas_cols[0]
        pivot = df_te.groupby(['Horizonte','Algoritmo'])[_c].mean().unstack()
        print(pivot.round(4).to_string())

    df_resumo_alg = df_te.groupby('Algoritmo')[metricas_cols].mean().reset_index() if metricas_cols else pd.DataFrame()
    df_resumo_alg.to_csv(PASTA_SAIDA/'resumo_metricas_algoritmos.csv', index=False)
    logger.info('Tabela resumo de métricas salva')
else:
    print('⚠️  resultados_teste.csv não encontrado')

## Etapa 4 — Análise de Métricas por Setor

In [ ]:
mapa_setor = {}
if 'CNPJ_CIA' in dataset.columns and 'SETOR' in dataset.columns:
    mapa_setor = dataset.drop_duplicates('CNPJ_CIA').set_index('CNPJ_CIA')['SETOR'].to_dict()

rows_setor = []
df_setor   = pd.DataFrame()

if not df_pred.empty and mapa_setor:
    df_pred_s = df_pred.copy()
    df_pred_s['SETOR'] = df_pred_s['CNPJ_CIA'].map(mapa_setor)

    df_pred_best = df_pred_s[
        df_pred_s.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
    ].copy()
    df_pred_foco = df_pred_best[df_pred_best['Target'].isin(TARGETS_FOCO_TCC)]

    for (setor, target), grp in df_pred_foco.groupby(['SETOR','Target']):
        yt = grp['y_true'].values; yp = grp['y_pred'].values
        mask = np.isfinite(yt) & np.isfinite(yp)
        if mask.sum() < 2: continue
        m = calcular_metricas(yt[mask], yp[mask])
        base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
        horiz = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')
        rows_setor.append({'SETOR':setor,'Target':target,'Base':base,'Horizonte':horiz,
                           'Algoritmo':melhores.get(target,'?'), **m})

    if rows_setor:
        df_setor = pd.DataFrame(rows_setor)
        df_setor.to_csv(PASTA_SAIDA/'metricas_por_setor.csv', index=False)
        print('=== SMAPE médio por Setor × Horizonte (targets foco TCC) ===')
        pivot_s = df_setor.groupby(['SETOR','Horizonte'])['SMAPE'].mean().unstack()
        print(pivot_s.round(4).to_string())
        logger.info('Análise por setor: %d combinações', len(df_setor))
    else:
        print('⚠️  Sem combinações setor×target válidas')
else:
    print('⚠️  Predições detalhadas ou mapa de setor indisponíveis')

## Etapa 5 — Z-Score de Altman Adaptado para Mercados Emergentes (Z'')

`Z'' = 6.56·X1 + 3.26·X2 + 6.72·X3 + 1.05·X4`

| Variável | Definição |
|----------|-----------|
| X1 | Capital de Giro / Ativo Total |
| X2 | PL / Ativo Total (proxy: Lucros Retidos) |
| X3 | EBIT / Ativo Total |
| X4 | PL / Passivo Total |

Zonas: **> 2.60** Segura · **1.10–2.60** Cinza · **< 1.10** Insolvência

In [ ]:
ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10

def classificar_zona(z):
    if pd.isna(z):        return 'N/D'
    if z > ZONA_SEGURA:   return 'Segura'
    if z >= ZONA_CINZA_INF: return 'Cinza'
    return 'Insolvência'

def calcular_altman_zpp(df_emp):
    """Calcula Z'' linha a linha com mapeamento flexível de colunas."""
    _col = lambda *ns: next((n for n in ns if n in df_emp.columns), None)
    col_ac  = _col('BPA_1.01','ativo_circulante')
    col_pc  = _col('BPP_2.01','passivo_circulante')
    col_at  = _col('BPA_1',   'ativo_total')
    col_pl  = _col('BPP_2',   'patrimonio_liquido')
    col_ebt = _col('EBITDA',  'ebitda','ebit')
    col_pnc = _col('BPP_2.03','passivo_nao_circulante')

    rows = []
    for _, row in df_emp.iterrows():
        try:
            at  = float(row[col_at])  if col_at  else np.nan
            ac  = float(row[col_ac])  if col_ac  else np.nan
            pc  = float(row[col_pc])  if col_pc  else np.nan
            pl  = float(row[col_pl])  if col_pl  else np.nan
            ebt = float(row[col_ebt])*0.85 if col_ebt and pd.notna(row.get(col_ebt,np.nan)) else np.nan
            pnc = float(row[col_pnc]) if col_pnc and pd.notna(row.get(col_pnc,np.nan)) else 0.0
            pt  = pc + pnc if pd.notna(pc) else np.nan

            X1 = (ac-pc)/at    if at>0 and pd.notna(ac) and pd.notna(pc) else np.nan
            X2 = pl/at         if at>0 and pd.notna(pl)                   else np.nan
            X3 = ebt/at        if at>0 and pd.notna(ebt)                  else np.nan
            X4 = pl/pt         if pd.notna(pt) and pt>0 and pd.notna(pl)  else np.nan

            ok = sum(pd.notna(v) for v in [X1,X2,X3,X4])
            z  = (6.56*(X1 or 0)+3.26*(X2 or 0)+6.72*(X3 or 0)+1.05*(X4 or 0)) if ok>=3 else np.nan
            rows.append({'X1_CG_AT':X1,'X2_PL_AT':X2,'X3_EBIT_AT':X3,'X4_PL_PT':X4,
                         'altman_z_pp':z,'zona_altman':classificar_zona(z),'comp_ok':ok})
        except Exception:
            rows.append({'X1_CG_AT':np.nan,'X2_PL_AT':np.nan,'X3_EBIT_AT':np.nan,
                         'X4_PL_PT':np.nan,'altman_z_pp':np.nan,'zona_altman':'N/D','comp_ok':0})
    return pd.DataFrame(rows, index=df_emp.index)

df_zscore = pd.DataFrame()
if 'CNPJ_CIA' in dataset.columns:
    dfs = []
    for cnpj, grp in dataset.groupby('CNPJ_CIA'):
        res  = calcular_altman_zpp(grp.sort_values('ANO') if 'ANO' in grp.columns else grp)
        meta = [c for c in ['CNPJ_CIA','NOME_CIA','ANO','SETOR','ORIGEM','DT_REFER'] if c in grp.columns]
        dfs.append(grp[meta].reset_index(drop=True).join(res.reset_index(drop=True)))
    df_zscore = pd.concat(dfs, ignore_index=True)
    df_zscore.to_csv(PASTA_SAIDA/'altman_zscore.csv', index=False)
    df_zscore.to_parquet(PASTA_SAIDA/'altman_zscore.parquet', index=False)

    df_zv = df_zscore[df_zscore['altman_z_pp'].notna()]
    n_tot = len(df_zv)
    print(f"=== Z'' de Altman — {n_tot:,} observações válidas ===")
    for zona, cnt in df_zv['zona_altman'].value_counts().items():
        print(f'  {zona:<15} {cnt:>5} ({cnt/n_tot*100:5.1f}%)')
    if 'ANO' in df_zscore.columns:
        print("\nZ'' médio por ano:")
        print(df_zv.groupby('ANO')['altman_z_pp'].agg(['mean','median','std']).round(3).to_string())
    if 'SETOR' in df_zscore.columns:
        print("\nZ'' médio por setor:")
        print(df_zv.groupby('SETOR')['altman_z_pp'].agg(['mean','median','std','count']).round(3).to_string())
    logger.info("Z'' calculado: %d obs válidas de %d total", n_tot, len(df_zscore))
else:
    print('⚠️  CNPJ_CIA não encontrado no dataset')

## Etapa 6 — Score de Risco Composto (8 KPIs, Limiares Consolidados)

In [ ]:
# (col, sentido, limiar, pontos, descrição)
REGRAS_RISCO = [
    ('liquidez_corrente', 'abaixo', 1.0,  15, 'Liquidez corrente < 1.0'),
    ('liquidez_imediata', 'abaixo', 0.3,  10, 'Liquidez imediata < 0.3'),
    ('margem_liquida',    'abaixo', 0.0,  20, 'Margem líquida negativa'),
    ('roe',               'abaixo', 0.0,  10, 'ROE negativo'),
    ('endividamento',     'acima',  0.7,  15, 'Endividamento > 70%'),
    ('alavancagem_de',    'acima',  3.0,  10, 'D/E > 3x'),
    ('cobertura_juros',   'abaixo', 1.5,  15, 'Cobertura de juros < 1.5x'),
    ('margem_ebitda',     'abaixo', 0.05,  5, 'Margem EBITDA < 5%'),
]

def calc_score_risco(row):
    s, alertas = 0, []
    for col, snt, lim, pts, desc in REGRAS_RISCO:
        if col not in row.index: continue
        v = row[col]
        if pd.isna(v): continue
        if (v<lim if snt=='abaixo' else v>lim):
            s += pts; alertas.append(f'{desc} ({v:.3f})')
    return min(s, 100), alertas

def cls_risco(s):
    if s<20: return 'Baixo'
    if s<40: return 'Moderado'
    if s<60: return 'Elevado'
    return 'Crítico'

kpi_cols_disp = [r[0] for r in REGRAS_RISCO if r[0] in dataset.columns]
if kpi_cols_disp:
    dataset['score_risco']  = dataset.apply(lambda r: calc_score_risco(r)[0], axis=1)
    dataset['classe_risco'] = dataset['score_risco'].apply(cls_risco)

    print(f'Score de risco calculado | KPIs usados: {kpi_cols_disp}')
    dist = dataset['classe_risco'].value_counts()
    for cl, cnt in dist.items():
        print(f'  {cl:<10} {cnt:>6} ({cnt/len(dataset)*100:5.1f}%)')
    if 'SETOR' in dataset.columns:
        print('\nScore médio de risco por setor:')
        print(dataset.groupby('SETOR')['score_risco'].agg(['mean','median','max']).round(1).to_string())
else:
    print('⚠️  KPIs de risco não encontrados no dataset')
    print(f'KPIs buscados : {[r[0] for r in REGRAS_RISCO]}')

## Etapa 7 — Análise Probabilística de Estresse (Monte Carlo)

In [ ]:
N_SIMULACOES      = 500
SIGMA_PERTURBACAO = 0.15   # 15 % do valor nominal
SEED_MC           = 42

EMPRESAS_ANCORA = {}
if 'NOME_CIA' in dataset.columns and 'SETOR' in dataset.columns:
    for setor, grp in dataset.groupby('SETOR'):
        EMPRESAS_ANCORA[grp.groupby('NOME_CIA').size().idxmax()] = setor
    print(f'Empresas âncora: {list(EMPRESAS_ANCORA.keys())}')

def ultimos_kpis(nome):
    if 'NOME_CIA' not in dataset.columns: return None, None
    df_e = dataset[dataset['NOME_CIA']==nome]
    df_e = df_e.sort_values('ANO') if 'ANO' in df_e.columns else df_e
    if df_e.empty: return None, None
    u = df_e.iloc[-1]
    return u, (int(u['ANO']) if 'ANO' in u.index else None)

resultados_stress = []
rng = np.random.default_rng(SEED_MC)
TARGETS_STRESS = [t for t in TARGETS if t.endswith('_DFP')]

for empresa, setor in EMPRESAS_ANCORA.items():
    ult, ano_base = ultimos_kpis(empresa)
    if ult is None: continue

    for target in TARGETS_STRESS:
        if target not in melhores: continue
        alg_nome = melhores[target]
        feats_t  = [f for f in selected_features_por_target.get(target,[]) if f in dataset.columns]
        if not feats_t: continue

        cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
        if not cam.exists(): continue
        try:
            obj = joblib.load(cam)
            modelo = obj['modelo'] if isinstance(obj,dict) else obj
        except Exception: continue

        tr = get_target_transform(target)
        x_base = np.array([float(ult.get(f,0.0) or 0.0) for f in feats_t])
        ruido  = rng.normal(0, SIGMA_PERTURBACAO*(np.abs(x_base)+1e-9),
                            size=(N_SIMULACOES, len(feats_t)))
        X_sim  = x_base[None,:] + ruido
        try:
            y_pred = target_inverse_transform(modelo.predict(X_sim), tr)
            y_base = float(target_inverse_transform(modelo.predict(x_base[None,:]), tr)[0])
        except Exception: continue

        p5,p50,p95 = np.percentile(y_pred,[5,50,95])
        base_nome  = target.replace('TARGET_','').replace('_DFP','')
        resultados_stress.append({
            'empresa':empresa,'setor':setor,'ano_base':ano_base,'target':target,
            'indicador':NOME_TARGET.get(base_nome,base_nome),'algoritmo':alg_nome,
            'y_base':y_base,'media_mc':float(np.mean(y_pred)),'std_mc':float(np.std(y_pred)),
            'p5':p5,'p50':p50,'p95':p95,
            'cv':float(np.std(y_pred)/abs(np.mean(y_pred))) if abs(np.mean(y_pred))>1e-9 else np.nan,
            'p_negativo':float(np.mean(y_pred<0)) if any(s in target for s in ['DRE_3.11','DFC']) else np.nan,
            'n_sim':N_SIMULACOES,'sigma':SIGMA_PERTURBACAO,
        })

if resultados_stress:
    df_stress = pd.DataFrame(resultados_stress)
    df_stress.to_csv(PASTA_SAIDA/'analise_estresse_mc.csv', index=False)
    print(f'=== Estresse Monte Carlo ({N_SIMULACOES} sim) — {len(df_stress)} combinações ===')
    cols = [c for c in ['empresa','indicador','y_base','media_mc','std_mc','p5','p50','p95','cv']
            if c in df_stress.columns]
    print(df_stress[cols].to_string(index=False, float_format='{:,.0f}'.format))
    logger.info('Estresse MC: %d combinações empresa×target', len(df_stress))
else:
    df_stress = pd.DataFrame()
    print('⚠️  Simulação de estresse não executada (modelos ou KPIs indisponíveis)')

## Etapa 8 — Feature Importance Agregada e Rankings

In [ ]:
df_feat_rank = pd.DataFrame()

if feature_importances:
    all_imp = {}
    for target, info in feature_importances.items():
        if not info or 'importancias' not in info: continue
        for feat, val in info['importancias'].items():
            if pd.isna(val): continue
            all_imp[feat] = all_imp.get(feat, 0.0) + float(val)

    df_feat_rank = (pd.DataFrame.from_dict(all_imp,'index',columns=['importancia_total'])
                    .sort_values('importancia_total',ascending=False)
                    .reset_index().rename(columns={'index':'feature'}))
    df_feat_rank['rank'] = df_feat_rank.index + 1

    def familia(f):
        if f.startswith('macro_'):    return 'Macro'
        if '_lag' in f or '_roll' in f: return 'Lag/Roll'
        if '_yoy' in f:                return 'YoY'
        if 'setor_' in f:              return 'Setor dummy'
        if '_x_' in f.lower():         return 'Interação setor'
        if f in (KPIS or []):          return 'KPI base'
        if f in ['flag_covid','ano_norm']: return 'Temporal'
        return 'Outro'

    df_feat_rank['familia'] = df_feat_rank['feature'].apply(familia)
    df_feat_rank.to_csv(PASTA_SAIDA/'feature_importance_ranking.csv', index=False)

    print('=== Top-20 Features por Importância Agregada ===')
    print(df_feat_rank.head(20)[['rank','feature','familia','importancia_total']].to_string(index=False))
    print('\nImportância por família:')
    print(df_feat_rank.groupby('familia')['importancia_total']
          .agg(['sum','count','mean']).round(4).to_string())
    logger.info('Feature importance agregada: %d features', len(df_feat_rank))
else:
    print('⚠️  feature_importances.pkl não disponível')

## Etapa 9 — Análise de Resíduos por Empresa e Horizonte

In [ ]:
if not df_pred.empty:
    df_res = df_pred[
        df_pred.apply(lambda r: melhores.get(r['Target'])==r['Algoritmo'], axis=1)
    ].copy()

    if not df_res.empty:
        df_res['erro_rel'] = (
            (df_res['y_true']-df_res['y_pred']) /
            df_res['y_true'].abs().clip(lower=1e-9)
        )
        df_res['horizonte'] = df_res['Target'].apply(
            lambda t: next((h for h in _HORIZONTES if t.endswith(h)), 'N/A'))
        df_res['base_target'] = df_res['Target'].apply(
            lambda t: t.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0])

        print('=== Estatísticas de Resíduos Relativos (melhor modelo por target) ===')
        stats_res = df_res.groupby(['base_target','horizonte'])['erro_rel'].agg(
            media='mean', mediana='median', std='std',
            p5=lambda x: np.percentile(x.dropna(),5),
            p95=lambda x: np.percentile(x.dropna(),95),
            skew=lambda x: x.dropna().skew()
        ).round(4)
        print(stats_res.to_string())

        if 'CNPJ_CIA' in df_res.columns and 'NOME_CIA' in dataset.columns:
            mapa_nome = dataset.drop_duplicates('CNPJ_CIA').set_index('CNPJ_CIA')['NOME_CIA'].to_dict()
            df_res['NOME_CIA'] = df_res['CNPJ_CIA'].map(mapa_nome)
            piores = (df_res[df_res['base_target'].isin(['DRE_3.01','DRE_3.11','EBITDA'])]
                      .groupby(['CNPJ_CIA','NOME_CIA'])['erro_rel']
                      .apply(lambda x: np.mean(np.abs(x.dropna())))
                      .nlargest(5))
            print('\nTop-5 empresas com maior MAPE médio (targets foco TCC):')
            print(piores.round(4).to_string())

        df_res.to_parquet(PASTA_SAIDA/'residuos_detalhados.parquet', index=False)
        logger.info('Resíduos detalhados: %d linhas', len(df_res))
else:
    print('⚠️  predicoes_teste_detalhadas.parquet não disponível')

## Etapa 10 — Visualizações

In [ ]:
PALETA_ALG = {'Ridge':'#3498db','SVR':'#9b59b6',
              'RandomForest':'#2ecc71','GradientBoosting':'#e74c3c','Ensemble':'#f39c12'}

# ── Fig 1: Heatmap SMAPE por algoritmo × horizonte ────────────────────────────
if not df_te.empty and 'Horizonte' in df_te.columns:
    try:
        _c = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
        if _c:
            pv = (df_te[df_te['Target'].isin(TARGETS_FOCO_TCC)]
                  .groupby(['Algoritmo','Horizonte'])[_c].mean().unstack())
            fig, ax = plt.subplots(figsize=(8,4))
            sns.heatmap(pv, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax,
                        linewidths=0.5, linecolor='white', cbar_kws={'label':'SMAPE médio'})
            ax.set_title('SMAPE médio por Algoritmo × Horizonte\n(Receita, Lucro e EBITDA)',
                         fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'heatmap_smape.png', dpi=150, bbox_inches='tight')
            plt.close()
            print('✅ heatmap_smape.png')
    except Exception as e:
        print(f'⚠️  Heatmap: {e}')

# ── Fig 2: Distribuição Z'' por setor ─────────────────────────────────────────
if not df_zscore.empty and 'SETOR' in df_zscore.columns and 'altman_z_pp' in df_zscore.columns:
    try:
        df_zp = df_zscore[df_zscore['altman_z_pp'].notna()]
        setores = sorted(df_zp['SETOR'].dropna().unique())
        fig, axes = plt.subplots(1, len(setores), figsize=(4*len(setores), 5), sharey=False)
        if len(setores)==1: axes=[axes]
        for ax, s in zip(axes, setores):
            dados = df_zp[df_zp['SETOR']==s]['altman_z_pp'].dropna()
            ax.hist(dados, bins=15, color='#3498db', edgecolor='white', alpha=0.85)
            ax.axvline(ZONA_CINZA_INF, color='#e74c3c', ls='--', lw=1.5)
            ax.axvline(ZONA_SEGURA,   color='#2ecc71', ls='--', lw=1.5)
            ax.set_title(s, fontsize=9, fontweight='bold')
            ax.set_xlabel("Z''", fontsize=8)
        plt.suptitle("Distribuição Z'' por Setor", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'dist_zscore_por_setor.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('✅ dist_zscore_por_setor.png')
    except Exception as e:
        print(f'⚠️  Dist Z-Score: {e}')

# ── Fig 3: Score de risco por setor (boxplot) ─────────────────────────────────
if 'score_risco' in dataset.columns and 'SETOR' in dataset.columns:
    try:
        df_rp = dataset[dataset['score_risco'].notna()]
        ord_s = df_rp.groupby('SETOR')['score_risco'].median().sort_values(ascending=False).index
        fig, ax = plt.subplots(figsize=(9,5))
        sns.boxplot(data=df_rp, x='SETOR', y='score_risco', order=ord_s,
                    palette='RdYlGn_r', ax=ax)
        for lim, cor, lbl in [(20,'#2ecc71','Baixo (20)'),(40,'#f39c12','Moderado (40)'),(60,'#e74c3c','Elevado (60)')]:
            ax.axhline(lim, ls=':', color=cor, lw=1.5, label=lbl)
        ax.set_title('Score de Risco Composto por Setor', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'score_risco_por_setor.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('✅ score_risco_por_setor.png')
    except Exception as e:
        print(f'⚠️  Score risco: {e}')

# ── Fig 4: Feature importance agregada (top-20) ───────────────────────────────
if not df_feat_rank.empty:
    try:
        top20 = df_feat_rank.head(20)
        fam_cores = {'Macro':'#e74c3c','Lag/Roll':'#3498db','YoY':'#2ecc71',
                     'KPI base':'#f39c12','Setor dummy':'#9b59b6',
                     'Interação setor':'#1abc9c','Temporal':'#95a5a6','Outro':'#bdc3c7'}
        cores = top20['familia'].map(fam_cores).fillna('#bdc3c7')
        fig, ax = plt.subplots(figsize=(10,7))
        ax.barh(range(len(top20)), top20['importancia_total'].values[::-1],
                color=cores.values[::-1])
        ax.set_yticks(range(len(top20)))
        ax.set_yticklabels(top20['feature'].values[::-1], fontsize=9)
        ax.set_xlabel('Importância Agregada', fontsize=10)
        ax.set_title('Top-20 Features — Importância Agregada (todos os targets)',
                     fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        patches = [mpatches.Patch(color=v, label=k) for k,v in fam_cores.items()]
        ax.legend(handles=patches, fontsize=8, loc='lower right')
        plt.tight_layout()
        plt.savefig(PASTA_SAIDA/'figuras'/'feature_importance_agregada.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('✅ feature_importance_agregada.png')
    except Exception as e:
        print(f'⚠️  Feature importance fig: {e}')

# ── Fig 5: Evolução Z'' das empresas âncora ───────────────────────────────────
if not df_zscore.empty and 'ANO' in df_zscore.columns and 'NOME_CIA' in df_zscore.columns:
    try:
        emps = list(EMPRESAS_ANCORA.keys())[:5]
        df_ze = df_zscore[df_zscore['NOME_CIA'].isin(emps) & df_zscore['altman_z_pp'].notna()]
        if not df_ze.empty:
            fig, ax = plt.subplots(figsize=(11,5))
            cores_e = plt.cm.Set2(np.linspace(0,1,len(emps)))
            for i,emp in enumerate(emps):
                sub = df_ze[df_ze['NOME_CIA']==emp].sort_values('ANO')
                if sub.empty: continue
                ax.plot(sub['ANO'], sub['altman_z_pp'], marker='o', label=emp,
                        color=cores_e[i], lw=2, ms=5)
            ax.axhspan(-5, ZONA_CINZA_INF, alpha=0.07, color='#e74c3c')
            ax.axhspan(ZONA_CINZA_INF, ZONA_SEGURA, alpha=0.07, color='#f39c12')
            ax.axhspan(ZONA_SEGURA, 20, alpha=0.05, color='#2ecc71')
            ax.axhline(ZONA_CINZA_INF, color='#e74c3c', ls='--', lw=1.2)
            ax.axhline(ZONA_SEGURA,   color='#2ecc71', ls='--', lw=1.2)
            ax.set_xlabel('Ano'); ax.set_ylabel("Z''")
            ax.set_title("Evolução Z'' — Empresas Representativas (2015–2025)",
                         fontsize=12, fontweight='bold')
            ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'evolucao_zscore.png', dpi=150, bbox_inches='tight')
            plt.close()
            print('✅ evolucao_zscore.png')
    except Exception as e:
        print(f'⚠️  Evolução Z-Score: {e}')

# ── Fig 6: Predito × Observado (foco TCC) ────────────────────────────────────
if not df_pred.empty:
    try:
        df_fp = df_pred[
            df_pred['Target'].isin(TARGETS_FOCO_TCC) &
            df_pred.apply(lambda r: melhores.get(r['Target'])==r['Algoritmo'], axis=1)
        ].copy()
        targets_plot = [t for t in TARGETS_FOCO_TCC if t in df_fp['Target'].values][:12]
        if targets_plot:
            ncols = 4; nrows = int(np.ceil(len(targets_plot)/ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
            axes = axes.flatten()
            for i, target in enumerate(targets_plot):
                ax = axes[i]
                sub = df_fp[df_fp['Target']==target].dropna(subset=['y_true','y_pred'])
                if sub.empty: ax.set_visible(False); continue
                yt, yp = sub['y_true'].values, sub['y_pred'].values
                alg = melhores.get(target,'?')
                r2v = r2_seguro(yt,yp); sp = smape(yt,yp)
                ax.scatter(yp, yt, alpha=0.5, s=20, edgecolors='none',
                           color=PALETA_ALG.get(alg,'#3498db'))
                lims = [min(yt.min(),yp.min()), max(yt.max(),yp.max())]
                ax.plot(lims, lims, 'k--', lw=1)
                base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
                horiz = next((h.replace('_','') for h in _HORIZONTES if target.endswith(h)),'')
                ax.set_title(f'{NOME_TARGET.get(base,base)} [{horiz}]\n{alg}  R²={r2v:.3f}  SMAPE={sp:.1%}',
                             fontsize=8)
                ax.set_xlabel('Predito', fontsize=7); ax.set_ylabel('Observado', fontsize=7)
                ax.tick_params(labelsize=7)
            for j in range(i+1, len(axes)): axes[j].set_visible(False)
            plt.suptitle('Predito × Observado — Targets Foco TCC (hold-out 2024–2025)',
                         fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.savefig(PASTA_SAIDA/'figuras'/'predito_vs_observado.png', dpi=150, bbox_inches='tight')
            plt.close()
            print('✅ predito_vs_observado.png')
    except Exception as e:
        print(f'⚠️  Predito vs Observado: {e}')

## Etapa 11 — Persistência Completa de Artefatos para o Script 5

In [ ]:
# ── melhores_modelos_v4.pkl ───────────────────────────────────────────────────
with open(PASTA_SAIDA/'melhores_modelos_v4.pkl','wb') as f:
    pickle.dump(melhores, f)

# ── zscore_por_empresa.pkl ────────────────────────────────────────────────────
if not df_zscore.empty:
    zpe = {}
    for cnpj, grp in df_zscore.dropna(subset=['altman_z_pp']).groupby('CNPJ_CIA'):
        grp_s = grp.sort_values('ANO') if 'ANO' in grp.columns else grp
        last  = grp_s.iloc[-1]
        zpe[cnpj] = {
            'z_medio':   float(grp['altman_z_pp'].mean()),
            'z_ultimo':  float(last['altman_z_pp']),
            'zona':      last['zona_altman'],
            'nome':      last.get('NOME_CIA', cnpj) if 'NOME_CIA' in last.index else cnpj,
            'setor':     last.get('SETOR', '')     if 'SETOR'    in last.index else '',
        }
    with open(PASTA_SAIDA/'zscore_por_empresa.pkl','wb') as f:
        pickle.dump(zpe, f)
    print(f'✅ zscore_por_empresa.pkl — {len(zpe)} empresas')

# ── score_risco_dataset.parquet ───────────────────────────────────────────────
if 'score_risco' in dataset.columns:
    cols_s = [c for c in ['CNPJ_CIA','NOME_CIA','ANO','SETOR',
                           'score_risco','classe_risco','altman_z_pp','zona_altman']
              if c in dataset.columns]
    dataset[cols_s].to_parquet(PASTA_SAIDA/'score_risco_dataset.parquet', index=False)
    print('✅ score_risco_dataset.parquet salvo')

# ── feature_importance_ranking.pkl ───────────────────────────────────────────
if not df_feat_rank.empty:
    with open(PASTA_SAIDA/'feature_importance_ranking.pkl','wb') as f:
        pickle.dump(df_feat_rank, f)

# ── relatorio_avaliacao_v4.json ───────────────────────────────────────────────
from collections import Counter
relatorio = {
    'versao': 'V4_AvaliacaoAprofundada',
    'data_execucao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_targets': len(TARGETS),
    'n_treino': int(len(treino)), 'n_teste': int(len(teste)),
    'melhores_modelos': {t:str(a) for t,a in melhores.items()},
    'contagem_vitorias': dict(Counter(melhores.values())),
    'zscore_altman': {
        'formula': "Z'' = 6.56·X1 + 3.26·X2 + 6.72·X3 + 1.05·X4",
        'limiar_segura': ZONA_SEGURA, 'limiar_cinza': ZONA_CINZA_INF,
        'obs_validas': int(df_zscore['altman_z_pp'].notna().sum()) if not df_zscore.empty else 0,
        'zonas': df_zscore['zona_altman'].value_counts().to_dict() if not df_zscore.empty else {},
    },
    'score_risco': {
        'n_kpis': len(kpi_cols_disp), 'kpis': kpi_cols_disp,
        'classes': dataset['classe_risco'].value_counts().to_dict() if 'classe_risco' in dataset.columns else {},
    },
    'estresse_mc': {
        'n_sim': N_SIMULACOES, 'sigma': SIGMA_PERTURBACAO,
        'combinacoes': len(resultados_stress),
    },
    'feature_importance': {
        'n_features': len(df_feat_rank) if not df_feat_rank.empty else 0,
        'top10': df_feat_rank.head(10)['feature'].tolist() if not df_feat_rank.empty else [],
        'familias': df_feat_rank.groupby('familia')['importancia_total'].sum().to_dict() if not df_feat_rank.empty else {},
    },
    'metricas_foco_tcc': {},
}
for t in TARGETS_FOCO_TCC:
    if t not in melhores or t not in metricas_teste_pkl: continue
    alg = melhores[t]; m = metricas_teste_pkl[t].get(alg,{})
    relatorio['metricas_foco_tcc'][t] = {
        'algoritmo': alg,
        'SMAPE':  float(m.get('SMAPE_macro_empresa', np.nan)),
        'RMSE':   float(m.get('RMSE_macro_empresa',  np.nan)),
        'R2':     float(m.get('R2_macro_empresa',    np.nan)),
        'TheilU': float(m.get('TheilU_macro_empresa',np.nan)),
        'DA':     float(m.get('DA_macro_empresa',    np.nan)),
    }

with open(PASTA_SAIDA/'logs'/'relatorio_avaliacao_v4.json','w',encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

# ── Resumo final ──────────────────────────────────────────────────────────────
print('\n' + '═'*90)
print('RESUMO FINAL — Script 4: Avaliação Aprofundada + Risco Corporativo')
print('═'*90)
print(f'  Targets avaliados    : {len(TARGETS)}')
print(f'  Targets foco TCC     : {len(TARGETS_FOCO_TCC)} (Receita, Lucro, EBITDA × 4 horizontes)')
dom = max(Counter(melhores.values()), key=lambda k: Counter(melhores.values())[k])
print(f'  Modelo dominante     : {dom} ({Counter(melhores.values())[dom]}× melhor)')
if not df_zscore.empty and 'altman_z_pp' in df_zscore.columns:
    nz = df_zscore['altman_z_pp'].notna().sum()
    ni = (df_zscore['zona_altman']=='Insolvência').sum()
    print(f"  Z'' de Altman        : {nz:,} obs | {ni:,} em zona de insolvência ({ni/nz*100:.1f}%)")
if resultados_stress:
    print(f'  Estresse Monte Carlo : {len(df_stress)} combinações empresa×target ({N_SIMULACOES} sim cada)')
print('─'*90)
print('Artefatos gerados em outputs/:')
for arq in ['altman_zscore.{csv,parquet}','score_risco_dataset.parquet',
            'analise_estresse_mc.csv','metricas_por_setor.csv',
            'feature_importance_ranking.{csv,pkl}','residuos_detalhados.parquet',
            'melhores_modelos_v4.pkl','zscore_por_empresa.pkl',
            'figuras/*.png (6 gráficos)','logs/relatorio_avaliacao_v4.json']:
    print(f'  - {arq}')
print('═'*90)
print('✅ Pronto para o Script 5 (Cenários Estratégicos + LLM)')
print('═'*90)
logger.info('Script 4 concluído. %d targets avaliados.', len(TARGETS))